## TEAM: Manal Mehdaoui, Ghita Sbai, Younes Barich

# MM-CTR Challenge – Solution Notebook
### Architecture: Sequential Transformer + DCNv2 (inspired on 1st place winning approach)
- **Task 1**: Contrastive multimodal embedding (BERT + CLIP fusion)
- **Task 2**: TransformerDCN on official embeddings
- **Task 1&2**: TransformerDCN on our Task1 embeddings

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os

BASE          = "/content/drive/MyDrive/mmctr"
DATA_DIR      = f"{BASE}/data"
MICROLENS_DIR = f"{DATA_DIR}/MicroLens_1M_x1"
os.makedirs(f"{BASE}/checkpoints", exist_ok=True)
os.makedirs(f"{BASE}/submission",  exist_ok=True)
print("Base:", BASE)
print("data/:", sorted(os.listdir(DATA_DIR)))
print("MicroLens/:", sorted(os.listdir(MICROLENS_DIR)))

Mounted at /content/drive
Base: /content/drive/MyDrive/mmctr
data/: ['MicroLens_1M_x1', 'README', 'item_emb.parquet', 'item_feature.parquet', 'item_images.rar', 'item_seq.parquet']
MicroLens/: ['item_info.parquet', 'item_info_task1.parquet', 'item_info_v3.parquet', 'test.parquet', 'train.parquet', 'valid.parquet']


In [ ]:
!pip install -q fuxictr==2.3.7 pyarrow pandas tqdm scikit-learn
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, pandas as pd, random, os
from tqdm import tqdm

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")
print("Device:", DEVICE)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.1/88.1 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.6/42.6 kB 4.5 MB/s eta 0:00:00
PyTorch: 2.10.0+cu128
GPU: NVIDIA A100-SXM4-80GB
Device: cuda


## Task 1 · Multimodal Embedding Model
Gated BERT+CLIP fusion trained with InfoNCE contrastive loss on co-click pairs.

In [ ]:
import random, logging
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader

class CrossModalGate(nn.Module):
    def __init__(self, text_dim, img_dim, hidden_dim):
        super().__init__()
        self.text_proj = nn.Linear(text_dim, hidden_dim)
        self.img_proj  = nn.Linear(img_dim,  hidden_dim)
        self.gate = nn.Sequential(
            nn.Linear(text_dim + img_dim, hidden_dim),
            nn.ReLU(), nn.Linear(hidden_dim, 1), nn.Sigmoid()
        )
    def forward(self, t, v):
        g = self.gate(torch.cat([t, v], dim=-1))
        return g * self.text_proj(t) + (1 - g) * self.img_proj(v)

class MultimodalEncoder(nn.Module):
    def __init__(self, text_dim, img_dim, hidden_dim=512, output_dim=128):
        super().__init__()
        self.fusion = CrossModalGate(text_dim, img_dim, hidden_dim)
        self.bn     = nn.BatchNorm1d(hidden_dim)
        self.proj   = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),
            nn.Dropout(0.1), nn.Linear(hidden_dim, output_dim)
        )
    def forward(self, t, v):
        return F.normalize(self.proj(self.bn(self.fusion(t, v))), p=2, dim=-1)

def info_nce_loss(anchor, positive, temperature=0.07):
    B = anchor.shape[0]
    sim = torch.mm(anchor, positive.T) / temperature
    return F.cross_entropy(sim, torch.arange(B, device=anchor.device))

class CoOccurrenceDataset(Dataset):
    def __init__(self, seq_df, seq_window=5, max_pairs=5_000_000):
        seq_col = [c for c in seq_df.columns if "item" in c.lower() or "seq" in c.lower()][0]
        pairs = []
        for _, row in tqdm(seq_df.iterrows(), total=len(seq_df), desc="Building pairs"):
            seq = row[seq_col]
            if isinstance(seq, np.ndarray): seq = seq.tolist()
            seq = [int(x) for x in seq if int(x) > 0]
            for i in range(len(seq)):
                for j in range(i+1, min(i+seq_window+1, len(seq))):
                    pairs.append((seq[i], seq[j]))
                if len(pairs) >= max_pairs: break
            if len(pairs) >= max_pairs: break
        random.shuffle(pairs)
        self.pairs = pairs
        print(f"Total pairs: {len(self.pairs)}")
    def __len__(self): return len(self.pairs)
    def __getitem__(self, idx): return self.pairs[idx]

print("Task 1 model classes ready")

Task 1 model classes ready


## Task 1 · Train Embedding Model
⚠️ **Skip this cell if `item_info_task1.parquet` already exists on your Drive.**

In [ ]:
# ── SKIP THIS CELL IF TASK 1 ALREADY TRAINED ────────────────────────────────
CKPT_T1 = f"{BASE}/checkpoints/best_embedding_model.pt"

if os.path.exists(CKPT_T1):
    print(f"Checkpoint already exists: {CKPT_T1}")
    print("Loading existing model — skipping training.")
    feat_df   = pd.read_parquet(f"{DATA_DIR}/item_feature.parquet")
    item_ids  = feat_df["item_id"].values
    bert_col  = [c for c in feat_df.columns if "bert" in c.lower() or "txt_emb" in c.lower()][0]
    clip_col  = [c for c in feat_df.columns if "clip" in c.lower() or "img_emb" in c.lower()][0]
    bert_embs = np.stack(feat_df[bert_col].values).astype(np.float32)
    clip_embs = np.stack(feat_df[clip_col].values).astype(np.float32)
    enc_model = MultimodalEncoder(bert_embs.shape[1], clip_embs.shape[1], 512, 128).to(DEVICE)
    enc_model.load_state_dict(torch.load(CKPT_T1, map_location=DEVICE))
    print("Task 1 model loaded.")
else:
    print("Training Task 1 embedding model...")
    EPOCHS_T1 = 15; BATCH_T1 = 1024; LR_T1 = 1e-3; TEMPERATURE = 0.07; SEED = 42
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
    feat_df   = pd.read_parquet(f"{DATA_DIR}/item_feature.parquet")
    item_ids  = feat_df["item_id"].values
    bert_col  = [c for c in feat_df.columns if "bert" in c.lower() or "txt_emb" in c.lower()][0]
    clip_col  = [c for c in feat_df.columns if "clip" in c.lower() or "img_emb" in c.lower()][0]
    bert_embs = np.stack(feat_df[bert_col].values).astype(np.float32)
    clip_embs = np.stack(feat_df[clip_col].values).astype(np.float32)
    id2idx    = {int(iid): i for i, iid in enumerate(item_ids)}
    seq_df    = pd.read_parquet(f"{DATA_DIR}/item_seq.parquet")
    dataset   = CoOccurrenceDataset(seq_df, seq_window=5, max_pairs=5_000_000)
    loader    = DataLoader(dataset, batch_size=BATCH_T1, shuffle=True, num_workers=2, drop_last=True)
    enc_model = MultimodalEncoder(bert_embs.shape[1], clip_embs.shape[1], 512, 128).to(DEVICE)
    opt_t1    = torch.optim.AdamW(enc_model.parameters(), lr=LR_T1, weight_decay=1e-5)
    sch_t1    = torch.optim.lr_scheduler.CosineAnnealingLR(opt_t1, T_max=EPOCHS_T1)
    bert_t    = torch.tensor(bert_embs)
    clip_t    = torch.tensor(clip_embs)
    best_loss = float("inf")
    for epoch in range(EPOCHS_T1):
        enc_model.train(); total, nb = 0.0, 0
        for batch in tqdm(loader, desc=f"Epoch {epoch+1}/{EPOCHS_T1}"):
            a_ids, p_ids = batch
            valid = [(a.item(), p.item()) for a, p in zip(a_ids, p_ids) if a.item() in id2idx and p.item() in id2idx]
            if len(valid) < 2: continue
            ai = [id2idx[a] for a,_ in valid]; pi = [id2idx[p] for _,p in valid]
            a_emb = enc_model(bert_t[ai].to(DEVICE), clip_t[ai].to(DEVICE))
            p_emb = enc_model(bert_t[pi].to(DEVICE), clip_t[pi].to(DEVICE))
            loss  = info_nce_loss(a_emb, p_emb, TEMPERATURE)
            opt_t1.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(enc_model.parameters(), 1.0); opt_t1.step()
            total += loss.item(); nb += 1
        avg = total / max(nb, 1); sch_t1.step()
        print(f"Epoch {epoch+1} | Loss: {avg:.4f}")
        if avg < best_loss:
            best_loss = avg; torch.save(enc_model.state_dict(), CKPT_T1)
            print(f"  → Best saved (loss={best_loss:.4f})")
    print("Task 1 training done!")

Checkpoint already exists: /content/drive/MyDrive/mmctr/checkpoints/best_embedding_model.pt
Loading existing model — skipping training.
Task 1 model loaded.


## Task 1 · Save Embeddings
⚠️ **Skip if `item_info_task1.parquet` already exists.**

In [ ]:
T1_INFO_PATH = f"{MICROLENS_DIR}/item_info_task1.parquet"

if os.path.exists(T1_INFO_PATH):
    print(f"Already exists: {T1_INFO_PATH} — skipping.")
else:
    enc_model.eval()
    bert_t = torch.tensor(bert_embs); clip_t = torch.tensor(clip_embs)
    all_embs = []
    with torch.no_grad():
        for start in tqdm(range(0, len(item_ids), 1024)):
            b = bert_t[start:start+1024].to(DEVICE)
            c = clip_t[start:start+1024].to(DEVICE)
            all_embs.append(enc_model(b, c).cpu().numpy().astype(np.float32))
    all_embs    = np.concatenate(all_embs, axis=0)
    new_emb_map = {int(iid): all_embs[i] for i, iid in enumerate(item_ids)}
    info_df     = pd.read_parquet(f"{MICROLENS_DIR}/item_info.parquet")
    emb_col     = [c for c in info_df.columns if "emb" in c.lower()][0]
    info_out    = info_df.copy()
    for i in range(1, len(info_out)):
        iid = int(info_out.at[i, "item_id"])
        if iid in new_emb_map:
            info_out.at[i, emb_col] = new_emb_map[iid]
    info_out[emb_col] = info_out[emb_col].apply(lambda x: np.array(x, dtype=np.float32))
    info_out.to_parquet(T1_INFO_PATH, index=False)
    print(f"Saved: {T1_INFO_PATH}")

Already exists: /content/drive/MyDrive/mmctr/data/MicroLens_1M_x1/item_info_task1.parquet — skipping.


## Shared Helpers · SeqTransformerDCN Architecture
Adapted from 1st place solution (Team momo, AUC=0.9839).
Key differences from baseline:
- Uses **last K cols** of sequence (most recent history) instead of top-k attention
- **Max pooling** over full sequence concatenated with last-K output  
- Multimodal embedding concatenated directly to item embedding (frozen)
- `dim_feedforward=256`, `num_heads=1`, `first_k_cols=16`

In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score
from torch.utils.data import Dataset, DataLoader

# ── Load splits ───────────────────────────────────────────────────────────────
train_df = pd.read_parquet(f"{MICROLENS_DIR}/train.parquet")
valid_df  = pd.read_parquet(f"{MICROLENS_DIR}/valid.parquet")
test_df   = pd.read_parquet(f"{MICROLENS_DIR}/test.parquet")

user_col  = "user_id"
item_col  = "item_id"
hist_col  = "item_seq"
label_col = "label"
print(f"Columns: user={user_col}, item={item_col}, hist={hist_col}, label={label_col}")

# ── Encoders ──────────────────────────────────────────────────────────────────
all_users = pd.concat([train_df[user_col], valid_df[user_col], test_df[user_col]]).unique()
info_base = pd.read_parquet(f"{MICROLENS_DIR}/item_info.parquet")
all_items = info_base["item_id"].values.astype(int)

user_enc  = LabelEncoder().fit(all_users.astype(int))
item_enc  = LabelEncoder().fit(all_items)
NUM_USERS = len(user_enc.classes_)
NUM_ITEMS = len(item_enc.classes_)

# Fast O(1) lookup dicts
user_map = {int(v): i+1 for i, v in enumerate(user_enc.classes_)}
item_map = {int(v): i+1 for i, v in enumerate(item_enc.classes_)}

def encode_id(val, enc, unk=0):
    m = user_map if enc is user_enc else item_map
    try:    return m.get(int(val), unk)
    except: return unk

# ── Tags (numpy int arrays of length 5) ──────────────────────────────────────
tag_col_info = "item_tags"
all_tag_ids  = []
for _, row in info_base.iterrows():
    t = row[tag_col_info]
    if isinstance(t, np.ndarray): all_tag_ids.extend(t.tolist())
NUM_TAGS = int(max(all_tag_ids)) + 2
MAX_TAG  = 5

item2tags = {}
for _, row in info_base.iterrows():
    iid = int(row["item_id"])
    t   = row[tag_col_info]
    item2tags[iid] = t.astype(int).tolist() if isinstance(t, np.ndarray) else [0]*MAX_TAG

MAX_SEQ = 64
print(f"Users={NUM_USERS} Items={NUM_ITEMS} Tags={NUM_TAGS} MAX_TAG={MAX_TAG} MAX_SEQ={MAX_SEQ}")

# ── CTR Dataset ───────────────────────────────────────────────────────────────
class CTRDataset(Dataset):
    def __init__(self, df, item2emb, mm_dim, has_label=True):
        self.df        = df.reset_index(drop=True)
        self.item2emb  = item2emb
        self.zero_emb  = np.zeros(mm_dim, dtype=np.float32)
        self.has_label = has_label

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row    = self.df.iloc[idx]
        uid    = encode_id(row[user_col], user_enc)
        raw_id = int(row[item_col])
        iid    = encode_id(raw_id, item_enc)
        mm     = self.item2emb.get(raw_id, self.zero_emb)
        tags   = item2tags.get(raw_id, [0]*MAX_TAG)

        hist_raw   = row[hist_col]
        hist_items = hist_raw.tolist() if isinstance(hist_raw, np.ndarray) else [0]*MAX_SEQ
        hist_items = hist_items[-MAX_SEQ:]
        pad        = MAX_SEQ - len(hist_items)
        # mask: True = PADDING (ignored), False = valid — matches PyTorch key_padding_mask convention
        mask       = [True]*pad + [False]*(MAX_SEQ - pad)
        hist_items = [0]*pad + hist_items

        hist_ids_enc = [encode_id(h, item_enc) for h in hist_items]
        hist_mm_list = [self.item2emb.get(int(h), self.zero_emb) for h in hist_items]

        likes = float(row.get("likes_level", 0)) / 10.0
        views = float(row.get("views_level", 0)) / 10.0

        return {
            "user_id":   torch.tensor(uid,  dtype=torch.long),
            "item_id":   torch.tensor(iid,  dtype=torch.long),
            "item_tags": torch.tensor(tags, dtype=torch.long),
            "item_mm":   torch.tensor(mm,   dtype=torch.float32),
            "hist_ids":  torch.tensor(hist_ids_enc, dtype=torch.long),
            "hist_mm":   torch.tensor(np.array(hist_mm_list, dtype=np.float32)),
            "hist_mask": torch.tensor(mask, dtype=torch.bool),
            "extra":     torch.tensor([likes, views], dtype=torch.float32),
            "label":     torch.tensor(int(row[label_col]) if self.has_label else 0, dtype=torch.float32),
        }

# ──  Transformer (last-K + max-pool) ─────────────────────────────────
class SeqTransformer(nn.Module):
    def __init__(self, item_dim, num_heads=1, num_layers=2, dropout=0.2,
                 dim_feedforward=256, first_k_cols=16, concat_max_pool=True):
        super().__init__()
        self.first_k_cols    = first_k_cols
        self.concat_max_pool = concat_max_pool
        d = item_dim * 2   # concat(hist_item, target) at each position
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d, nhead=num_heads, dim_feedforward=dim_feedforward,
            dropout=dropout, batch_first=True
        )
        self.tf      = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        if concat_max_pool:
            self.out_proj = nn.Linear(d, d)

    def forward(self, sequence_emb, target_emb, mask):
        # mask: True = padding  (PyTorch key_padding_mask convention)
        B, L, D = sequence_emb.shape
        # Ensure not all positions masked (would cause NaN in attention)
        fully_masked = mask.all(dim=-1)
        mask[fully_masked, -1] = False

        tgt    = target_emb.unsqueeze(1).expand(-1, L, -1)
        x      = torch.cat([sequence_emb, tgt], dim=-1)   # (B, L, 2D)
        x      = self.tf(x, src_key_padding_mask=mask)
        x      = x.masked_fill(mask.unsqueeze(-1), 0.)

        # last first_k_cols positions (most recent history)
        out_parts = [x[:, -self.first_k_cols:].flatten(start_dim=1)]

        if self.concat_max_pool:
            x_for_pool = x.masked_fill(mask.unsqueeze(-1), -1e9)
            pooled     = self.out_proj(x_for_pool.max(dim=1).values)
            out_parts.append(pooled)

        return torch.cat(out_parts, dim=-1)

# ── CrossNetV2 ────────────────────────────────────────────────────────────────
class CrossNetV2(nn.Module):
    def __init__(self, dim, n):
        super().__init__()
        self.layers = nn.ModuleList([nn.Linear(dim, dim) for _ in range(n)])
    def forward(self, x0):
        xl = x0
        for l in self.layers: xl = x0 * l(xl) + xl
        return xl

# ── MLP Block ─────────────────────────────────────────────────────────────────
class MLPBlock(nn.Module):
    def __init__(self, dims, dropout=0.2):
        super().__init__()
        mods = []
        for i in range(len(dims)-1):
            mods += [nn.Linear(dims[i], dims[i+1]),
                     nn.BatchNorm1d(dims[i+1]), nn.ReLU(), nn.Dropout(dropout)]
        self.net = nn.Sequential(*mods)
    def forward(self, x): return self.net(x)

# ── SeqTransformerDCN ──────────────────────────────────────────────────────
class SeqTransformerDCN(nn.Module):
    def __init__(self, num_users, num_items, num_tags, mm_dim=128, emb_dim=64,
                 num_heads=1, tf_layers=2, dim_feedforward=256, first_k_cols=16,
                 cross_num=3, dnn_units=[1024,512,256], pred_units=[64,32],
                 dropout=0.2, concat_max_pool=True):
        super().__init__()
        self.user_emb = nn.Embedding(num_users+1, emb_dim, padding_idx=0)
        self.item_emb = nn.Embedding(num_items+1, emb_dim, padding_idx=0)
        self.tag_emb  = nn.Embedding(num_tags+1,  emb_dim, padding_idx=0)
        # mm embedding concatenated directly to item emb (key design choice)
        self.mm_proj  = nn.Linear(mm_dim, emb_dim)
        # item_dim = emb_dim (item_id) + emb_dim (mm) = 2*emb_dim
        # but we keep item repr as emb_dim for sequence, concat mm separately
        item_seq_dim  = emb_dim   # item_id emb only for sequence
        self.seq_tf   = SeqTransformer(
            item_seq_dim, num_heads, tf_layers, dropout,
            dim_feedforward, first_k_cols, concat_max_pool
        )
        seq_out_dim  = (first_k_cols + int(concat_max_pool)) * item_seq_dim * 2
        # static features: user + item + mm_proj + tags + likes + views
        static_dim   = emb_dim * 4 + 2
        total_dim    = static_dim + seq_out_dim
        self.cross   = CrossNetV2(total_dim, cross_num)
        self.dnn     = MLPBlock([total_dim] + dnn_units, dropout)
        pred_in      = total_dim + dnn_units[-1]
        pred_mods    = []
        dims = [pred_in] + pred_units
        for i in range(len(dims)-1):
            pred_mods += [nn.Linear(dims[i], dims[i+1]), nn.ReLU()]
        pred_mods   += [nn.Linear(pred_units[-1], 1), nn.Sigmoid()]
        self.pred    = nn.Sequential(*pred_mods)

    def forward(self, user_id, item_id, item_tags, item_mm,
              hist_ids, hist_mm, hist_mask, extra):
      u   = self.user_emb(user_id)                   # (B, E)
      ext = extra                                     # (B, 2) — likes, views already normalized

      # target repr
      id_e  = self.item_emb(item_id)                 # (B, E)
      tag_e = self.tag_emb(item_tags).mean(dim=1)    # (B, E)
      mm_e  = self.mm_proj(item_mm)                  # (B, E)
      target_repr = torch.cat([id_e, tag_e, mm_e], dim=-1)  # (B, 3E)

      # history repr (item_id emb only for sequence)
      h_id  = self.item_emb(hist_ids)                # (B, L, E)
      h_mm  = self.mm_proj(hist_mm)                  # (B, L, E)
      hist_repr = h_id                               # seq_tf expects item_seq_dim=emb_dim

      seq_out = self.seq_tf(hist_repr, id_e, hist_mask)
      x = torch.cat([u, ext, target_repr, seq_out], dim=-1)
      return self.pred(torch.cat([self.cross(x), self.dnn(x)], dim=-1)).squeeze(-1)

# ── Loaders & training helpers ────────────────────────────────────────────────
def make_loaders(item2emb, mm_dim, batch_size=1024):
    tr = CTRDataset(train_df, item2emb, mm_dim, has_label=True)
    va = CTRDataset(valid_df,  item2emb, mm_dim, has_label=True)
    te = CTRDataset(test_df,   item2emb, mm_dim, has_label=False)
    return (DataLoader(tr, batch_size=batch_size, shuffle=True,  num_workers=4, pin_memory=True, persistent_workers=True),
            DataLoader(va, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True, persistent_workers=True),
            DataLoader(te, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True, persistent_workers=True))

def train_ctr(model, train_loader, valid_loader, lr, epochs, patience, ckpt_path, weight_decay=1e-5):
    opt  = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    sch  = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='max', factor=0.5, patience=2)
    crit = nn.BCELoss()
    best_auc, no_imp = 0.0, 0

    def run_epoch(loader, train=True):
        model.train() if train else model.eval()
        total, preds, labels = 0.0, [], []
        ctx = torch.enable_grad() if train else torch.no_grad()
        with ctx:
            for b in tqdm(loader, desc="train" if train else "valid", leave=False):
                uid  = b["user_id"].to(DEVICE);   iid  = b["item_id"].to(DEVICE)
                tags = b["item_tags"].to(DEVICE);  mm   = b["item_mm"].to(DEVICE)
                hids = b["hist_ids"].to(DEVICE);   hmm  = b["hist_mm"].to(DEVICE)
                hmsk = b["hist_mask"].to(DEVICE);  ext  = b["extra"].to(DEVICE)
                lbl  = b["label"].to(DEVICE)
                pred = model(uid, iid, tags, mm, hids, hmm, hmsk, ext)
                loss = crit(pred, lbl)
                if train:
                    opt.zero_grad(); loss.backward()
                    nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    opt.step()
                total += loss.item()
                preds.extend(pred.detach().cpu().numpy())
                labels.extend(lbl.cpu().numpy())
        auc = roc_auc_score(labels, preds) if len(set(labels)) > 1 else 0.0
        return total/len(loader), auc

    for epoch in range(epochs):
        tr_loss, tr_auc = run_epoch(train_loader, train=True)
        va_loss, va_auc = run_epoch(valid_loader, train=False)
        sch.step(va_auc)
        print(f"Epoch {epoch+1:2d}/{epochs} | Train {tr_loss:.4f}/{tr_auc:.4f} | Valid {va_loss:.4f}/{va_auc:.4f}")
        if va_auc > best_auc:
            best_auc = va_auc; no_imp = 0
            torch.save(model.state_dict(), ckpt_path)
            print(f"  → Best saved AUC={best_auc:.4f}")
        else:
            no_imp += 1
            if no_imp >= patience:
                print(f"Early stopping at epoch {epoch+1}"); break
    return best_auc

def predict_ctr(model, loader, ckpt_path):
    model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
    model.eval(); preds = []
    with torch.no_grad():
        for b in tqdm(loader, desc="predicting"):
            uid  = b["user_id"].to(DEVICE);   iid  = b["item_id"].to(DEVICE)
            tags = b["item_tags"].to(DEVICE);  mm   = b["item_mm"].to(DEVICE)
            hids = b["hist_ids"].to(DEVICE);   hmm  = b["hist_mm"].to(DEVICE)
            hmsk = b["hist_mask"].to(DEVICE);  ext  = b["extra"].to(DEVICE)
            preds.extend(model(uid, iid, tags, mm, hids, hmm, hmsk, ext).cpu().numpy())
    return np.array(preds)

print("All helpers ready!")

Columns: user=user_id, item=item_id, hist=item_seq, label=label
Users=1000000 Items=91718 Tags=11741 MAX_TAG=5 MAX_SEQ=64
All helpers ready!


## Task 2 · Train on Official Embeddings
Uses `item_emb_d128_v3` (PCA concat of BERT+CLIP) — the same choice as the 1st place solution.

In [ ]:
emb_df     = pd.read_parquet(f"{DATA_DIR}/item_emb.parquet")
emb_col_t2 = "item_emb_d128_v3"
print(f"Available: {emb_df.columns.tolist()}")
print(f"Using: {emb_col_t2}")

item2emb_t2 = {int(row["item_id"]): np.array(row[emb_col_t2], dtype=np.float32)
               for _, row in emb_df.iterrows()}
MM_DIM_T2   = len(next(iter(item2emb_t2.values())))
print(f"Embedding dim: {MM_DIM_T2}, items: {len(item2emb_t2)}")

train_ld_t2, valid_ld_t2, test_ld_t2 = make_loaders(item2emb_t2, MM_DIM_T2, batch_size=1024)
print(f"Loaders: train={len(train_ld_t2)} valid={len(valid_ld_t2)} test={len(test_ld_t2)}")

model_t2 = SeqTransformerDCN(
    num_users=NUM_USERS, num_items=NUM_ITEMS, num_tags=NUM_TAGS,
    mm_dim=MM_DIM_T2, emb_dim=64, num_heads=1, tf_layers=2,
    dim_feedforward=256, first_k_cols=16, cross_num=3,
    dnn_units=[1024,512,256], pred_units=[64,32],
    dropout=0.2, concat_max_pool=True
).to(DEVICE)
print(f"Task 2 params: {sum(p.numel() for p in model_t2.parameters() if p.requires_grad):,}")

CKPT_T2   = f"{BASE}/checkpoints/best_ctr_task2.pt"
best_t2   = train_ctr(model_t2, train_ld_t2, valid_ld_t2,
                       lr=5e-4, epochs=30, patience=5, ckpt_path=CKPT_T2)
print(f"\nTask 2 best validation AUC: {best_t2:.4f}")

task2_preds = predict_ctr(model_t2, test_ld_t2, CKPT_T2)
print(f"Task 2 predictions: {len(task2_preds)} | range [{task2_preds.min():.4f}, {task2_preds.max():.4f}]")

Available: ['item_id', 'item_emb_d128_v1', 'item_emb_d128_v2', 'item_emb_d128_v3', 'item_emb_d128_e4']
Using: item_emb_d128_v3
Embedding dim: 128, items: 91717
Loaders: train=3516 valid=10 test=371


/tmp/ipykernel_572/1633877795.py:109: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.num_heads is odd
  self.tf      = nn.TransformerEncoder(enc_layer, num_layers=num_layers)


Task 2 params: 92,019,155


Epoch  1/30 | Train 0.2631/0.9324 | Valid 2.3345/0.5822
  → Best saved AUC=0.5822


Epoch  2/30 | Train 0.1488/0.9792 | Valid 2.0391/0.6356
  → Best saved AUC=0.6356


...

Epoch 28/30 | Train 0.0030/1.0000 | Valid 0.8532/0.9021


Epoch 29/30 | Train 0.0030/1.0000 | Valid 0.8517/0.9011


Epoch 30/30 | Train 0.0007/1.0000 | Valid 1.0257/0.8990

Task 2 best validation AUC: 0.9062


predicting: 100%|██████████| 371/371 [00:27<00:00, 13.44it/s]

Task 2 predictions: 379142 | range [0.0000, 1.0000]


## Task 1&2 · Train on Our Task1 Embeddings

In [ ]:
T1_INFO_PATH = f"{MICROLENS_DIR}/item_info_task1.parquet"
info_t1      = pd.read_parquet(T1_INFO_PATH)
emb_col_t1   = [c for c in info_t1.columns if "emb" in c.lower()][0]

item2emb_t12 = {int(row["item_id"]): np.array(row[emb_col_t1], dtype=np.float32)
                for _, row in info_t1.iterrows()}
MM_DIM_T12   = len(next(iter(item2emb_t12.values())))
print(f"Task1 embedding dim: {MM_DIM_T12}, items: {len(item2emb_t12)}")

train_ld_t12, valid_ld_t12, test_ld_t12 = make_loaders(item2emb_t12, MM_DIM_T12, batch_size=1024)

model_t12 = SeqTransformerDCN(
    num_users=NUM_USERS, num_items=NUM_ITEMS, num_tags=NUM_TAGS,
    mm_dim=MM_DIM_T12, emb_dim=64, num_heads=1, tf_layers=2,
    dim_feedforward=256, first_k_cols=16, cross_num=3,
    dnn_units=[1024,512,256], pred_units=[64,32],
    dropout=0.2, concat_max_pool=True
).to(DEVICE)
print(f"Task 1&2 params: {sum(p.numel() for p in model_t12.parameters() if p.requires_grad):,}")

CKPT_T12    = f"{BASE}/checkpoints/best_ctr_task12.pt"
best_t12    = train_ctr(model_t12, train_ld_t12, valid_ld_t12,
                         lr=5e-4, epochs=30, patience=5, ckpt_path=CKPT_T12)
print(f"\nTask 1&2 best validation AUC: {best_t12:.4f}")

task12_preds = predict_ctr(model_t12, test_ld_t12, CKPT_T12)
print(f"Task1&2 predictions: {len(task12_preds)} | range [{task12_preds.min():.4f}, {task12_preds.max():.4f}]")

Task1 embedding dim: 128, items: 91718


/tmp/ipykernel_424/1633877795.py:109: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.num_heads is odd
  self.tf      = nn.TransformerEncoder(enc_layer, num_layers=num_layers)


Task 1&2 params: 92,019,155


Epoch  1/30 | Train 0.2562/0.9360 | Valid 1.7315/0.6156
  → Best saved AUC=0.6156


Epoch  2/30 | Train 0.1374/0.9824 | Valid 1.8686/0.6557
  → Best saved AUC=0.6557


...

Epoch 21/30 | Train 0.0037/1.0000 | Valid 0.6382/0.8891


Epoch 22/30 | Train 0.0015/1.0000 | Valid 0.8090/0.8902


Epoch 23/30 | Train 0.0010/1.0000 | Valid 0.8499/0.8902
Early stopping at epoch 23

Task 1&2 best validation AUC: 0.8938


predicting: 100%|██████████| 371/371 [00:28<00:00, 12.96it/s]

Task1&2 predictions: 379142 | range [0.0000, 1.0000]


## Build Submission CSV

In [ ]:
# Reload Task 2 predictions
CKPT_T2 = f"{BASE}/checkpoints/best_ctr_task2.pt"
model_t2 = SeqTransformerDCN(
    num_users=NUM_USERS, num_items=NUM_ITEMS, num_tags=NUM_TAGS,
    mm_dim=128, emb_dim=64, num_heads=1, tf_layers=2,
    dim_feedforward=256, first_k_cols=16, cross_num=3,
    dnn_units=[1024,512,256], pred_units=[64,32],
    dropout=0.2, concat_max_pool=True
).to(DEVICE)

emb_df = pd.read_parquet(f"{DATA_DIR}/item_emb.parquet")
item2emb_t2 = {int(row["item_id"]): np.array(row["item_emb_d128_v3"], dtype=np.float32)
               for _, row in emb_df.iterrows()}
_, _, test_ld_t2 = make_loaders(item2emb_t2, 128, batch_size=1024)
task2_preds = predict_ctr(model_t2, test_ld_t2, CKPT_T2)
print(f"Task 2 reloaded: {len(task2_preds)} predictions")

/tmp/ipykernel_1272/3884157588.py:109: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.num_heads is odd
  self.tf      = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
predicting: 100%|██████████| 371/371 [00:30<00:00, 12.09it/s]

Task 2 reloaded: 379142 predictions


In [ ]:
# Reload model_t12
CKPT_T12 = f"{BASE}/checkpoints/best_ctr_task12.pt"
model_t12 = SeqTransformerDCN(
    num_users=NUM_USERS, num_items=NUM_ITEMS, num_tags=NUM_TAGS,
    mm_dim=128, emb_dim=64, num_heads=1, tf_layers=2,
    dim_feedforward=256, first_k_cols=16, cross_num=3,
    dnn_units=[1024,512,256], pred_units=[64,32],
    dropout=0.2, concat_max_pool=True
).to(DEVICE)

T1_INFO_PATH = f"{MICROLENS_DIR}/item_info_task1.parquet"
info_t1 = pd.read_parquet(T1_INFO_PATH)
emb_col_t1 = [c for c in info_t1.columns if "emb" in c.lower()][0]
item2emb_t12 = {int(row["item_id"]): np.array(row[emb_col_t1], dtype=np.float32)
                for _, row in info_t1.iterrows()}
_, _, test_ld_t12 = make_loaders(item2emb_t12, 128, batch_size=1024)
print("model_t12 ready")

/tmp/ipykernel_1272/3884157588.py:109: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.num_heads is odd
  self.tf      = nn.TransformerEncoder(enc_layer, num_layers=num_layers)


model_t12 ready


In [ ]:
# ── INFERENCE TIME BENCHMARK ─────────────────────────────────────────────────
import time

NUM_TEST = len(test_df)

# Task 2
torch.cuda.synchronize() if torch.cuda.is_available() else None
t0 = time.time()
task2_preds_timed = predict_ctr(model_t2, test_ld_t2, CKPT_T2)
torch.cuda.synchronize() if torch.cuda.is_available() else None
t2_elapsed = time.time() - t0

# Task 1&2
torch.cuda.synchronize() if torch.cuda.is_available() else None
t0 = time.time()
task12_preds_timed = predict_ctr(model_t12, test_ld_t12, CKPT_T12)
torch.cuda.synchronize() if torch.cuda.is_available() else None
t12_elapsed = time.time() - t0

print(f'=== Inference Time Benchmark ===')
print(f'Test set size        : {NUM_TEST:,} samples')
print(f'Device               : {DEVICE} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"})')
print(f'Batch size           : 1024')
print(f'--- Task 2 ---')
print(f'Total time           : {t2_elapsed:.2f}s')
print(f'Throughput           : {NUM_TEST/t2_elapsed:,.0f} samples/sec')
print(f'Per-sample latency   : {t2_elapsed/NUM_TEST*1000:.3f} ms')
print(f'--- Task 1&2 ---')
print(f'Total time           : {t12_elapsed:.2f}s')
print(f'Throughput           : {NUM_TEST/t12_elapsed:,.0f} samples/sec')
print(f'Per-sample latency   : {t12_elapsed/NUM_TEST*1000:.3f} ms')

predicting: 100%|██████████| 371/371 [00:27<00:00, 13.42it/s]

=== Inference Time Benchmark ===
Test set size        : 379,142 samples
Device               : cuda (NVIDIA A100-SXM4-80GB)
Batch size           : 1024
--- Task 2 ---
Total time           : 27.24s
Throughput           : 13,918 samples/sec
Per-sample latency   : 0.072 ms
--- Task 1&2 ---
Total time           : 33.67s
Throughput           : 11,260 samples/sec
Per-sample latency   : 0.089 ms


In [ ]:
import shutil

task1_preds = task12_preds  # Task1 score = Task1&2 model

n = len(task2_preds)
assert len(task12_preds) == n, f"Length mismatch: task2={n}, task12={len(task12_preds)}"

submission = pd.DataFrame({
    "ID":      range(n),
    "Task1":   task1_preds,
    "Task2":   task2_preds,
    "Task1&2": task12_preds
})
CSV_PATH = f"{BASE}/submission/prediction.csv"
submission.to_csv(CSV_PATH, index=False)
print(f"Saved: {CSV_PATH}  ({len(submission)} rows)")
print(submission.head(5).to_string())

zip_base = f"{BASE}/submission/prediction"
shutil.make_archive(zip_base, 'zip', f"{BASE}/submission", "prediction.csv")
print(f"Zipped: {zip_base}.zip  ({os.path.getsize(zip_base+'.zip')/1024:.1f} KB)")

Saved: /content/drive/MyDrive/mmctr/submission/prediction.csv  (379142 rows)
   ID         Task1         Task2       Task1&2
0   0  1.439476e-01  6.175202e-01  1.439476e-01
1   1  9.992841e-01  9.873765e-01  9.992841e-01
2   2  8.354626e-01  4.533924e-01  8.354626e-01
3   3  1.925812e-08  1.010106e-15  1.925812e-08
4   4  7.183699e-09  7.205628e-20  7.183699e-09
Zipped: /content/drive/MyDrive/mmctr/submission/prediction.zip  (5392.5 KB)


In [ ]:
from google.colab import files
files.download(f"{BASE}/submission/prediction.zip")
print("Done! Upload prediction.zip to: https://www.codabench.org/competitions/5372/")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Done! Upload prediction.zip to: https://www.codabench.org/competitions/5372/
